# Jun-2026 LAPD run overview

Generic "what is in this run?" inspection — **no analysis happens here**.
Point `ifn` at any Jun-2026 datarun and read off: every scope's channel
descriptions, the operator's run description, the probe motion layout, the
merged interferometer traces, and which gas-puff group the runs fall into.

The analysis lives in the companion notebooks, which assume you already know
which scope/channel carries what:

* `Jun2026_IV_explore.ipynb` — swept Langmuir I-V → Vp / Te / ne
* `Jun2026_Isat_explore.ipynb` — fixed-bias Isat fluctuations + FFT
* `Jun2026_xcorr_explore.ipynb` — two-channel coherence / cross-phase

## 1. Configure — set the run file

In [1]:
import importlib
get_ipython().run_line_magic("matplotlib", "qt")
# get_ipython().run_line_magic("matplotlib", "inline")

import numpy as np
import matplotlib.pyplot as plt

import Jun2026_IV as jiv
importlib.reload(jiv)   # re-run this cell after editing Jun2026_IV.py

from data_analysis.io import (
    open_lapd, list_all_channels, print_run_description, read_interferometer)

# >>> SET THIS to the run you want to inspect <<<
ifn = r"D:\data\LAPD\jun2026-jia\00-He-800G-bias40V-LP-p29-line_2026-06-10.hdf5"

assert ifn, "Set `ifn` to a run file first."
run = open_lapd(ifn)
print("backend:", run.backend)

backend: pydaq


## 2. Channel descriptions + run description

Print every scope group's channel descriptions and the run's hand-written
description (plasma / bias / probe settings). Read these to decide which
scope + channel carries the signal you want — the analysis notebooks take the
selection by hand, nothing is guessed.

In [2]:
channels = list_all_channels(ifn)
print()
print_run_description(ifn);

Scope groups and channel descriptions:

  scope 'biasscope':
    C1: "Current on 2''"
    C2: "Current on 4''"
    C3: "Current on 6''"
    C4: "Bias voltage, 2'', X200"
    C5: "Bias voltage, 4'', X200"
    C6: "Bias voltage, 6'', X200"
    C7: "Bias voltage, 8'', X200"
    C8: 'Light @P30'

  scope 'lpscope':
    C2: 'V, LP@P29-L'
    C3: 'I, LP@P29-R'
    C4: 'V, LP@P29-R'

  scope 'machscope':
    C2: 'MP@P33, Isat-X-'
    C3: 'MP@P33, Isat-Y+'
    C4: 'MP@P33, Isat-Y-'

=== Run description ===
Documenting instability observed during electrode biasing with LP line and changing gas puff.

Operator: Jia Han, PP, Zoe
    
Setup:
    - Plasma condition: 
      - Heater 1860A,  bank 90V, anode-cathode 50V, current 4kA
      - Helium backside pressure 40 Psi, Puff voltage 75V for 24ms West+East
      - Discharge 20 ms, Pulsing 1/3 Hz, plasma breakdown ~10 ms
      - Pressure unknown
      - Port 20, density at 10 ms:  0.7e13 cm^-3
      - Port 29, density at 10 ms:  1e13 cm^-3
    - Magn

## 3. Probe positions

Motion-group layout: how many positions the probe drive visited and how many
shots were taken at each. For multi-drive runs set `motion_group_name`
(`read_lp_positions` lists the groups in its error).

In [3]:
motion_group_name = None    # e.g. "<Hermes>    p29_LP"

# Prints the motion-group layout summary itself; the analysis notebooks unpack
# its return values, here the printout is the point.
jiv.read_lp_positions(ifn, motion_group_name);

Using motion group: '<Hermes>    p29_LP'
Positions: 61 unique (x: 61, y: 1), 10 shots/position, 610 total shots.


## 4. Interferometer line-averaged density

`interf_merge_lapd_daq.py` (bapsf_interferometer repo) merges two interferometer
traces per channel into the datarun file — the trace nearest the **first** shot
and the one nearest the **last** shot — under `diagnostics/interferometer/`.
Phase [rad] is converted to line-averaged density with each channel's
`calibration factor (m^-3/rad)` attribute (assumes 40 cm plasma length) and
plotted in cm⁻³, one panel per chord.

The interferometer ne is a **chord average** along the beam path, so expect it
to sit below the peak local density a probe measures. The probe-vs-interferometer
density calibration lives in the IV notebook (its last section).

In [ ]:
# Merged interferometer traces (nearest first + last shot of the run), read via
# the shared reader; phase (rad) -> line-averaged ne via each channel's
# calibration attribute.
interf = read_interferometer(ifn)

fig, axes = plt.subplots(len(interf), 1, sharex=True, squeeze=False,
                         figsize=(10, 2.8 * len(interf)))

for ax, (name, ch) in zip(axes[:, 0], interf.items()):
    for shot, reason in ch.skipped.items():
        print(f"  {name} shot {shot}: skipped ({reason})")
    for ne_line, shot, when in zip(ch.ne_line_cm3, ch.shots, ch.when):
        ax.plot(ch.t_ms, ne_line, label=f"shot {shot} ({when})")
    ax.set_ylabel(r"$\bar{n}_e$ [cm$^{-3}$]")
    ax.set_title(name)
    if ch.shots:
        ax.legend()

axes[-1, 0].set_xlabel("t [ms]")
plt.tight_layout()
plt.show()

## 5. Runs grouped by gas-puff setting

Which runs share a gas-puff setting (written once by `group_by_gas_puff.py`
into `gas_puff_groups.npy` in the data dir) — the run-to-run comparison map
for the whole campaign, independent of the `ifn` above.

In [ ]:
# Runs grouped by gas-puff setting, written by group_by_gas_puff.py.
# {label: {"puff_v": volts, "puff_t": ms, "files": [hdf5 names...]}}
groups = np.load(
    r"D:\data\LAPD\jun2026-jia\gas_puff_groups.npy", allow_pickle=True
).item()

print(f"{len(groups)} gas-puff setting(s):")
for label, g in groups.items():
    print(f"\n  [{label}]  puff_v={g['puff_v']} V, puff_t={g['puff_t']} ms"
          f"  ({len(g['files'])} files)")
    for n in g["files"]:
        print(f"      {n}")

In [5]:
scope_name = "biasscope"
chan       = "C4"

data_arr, tarr = run.channel(chan, scope_name=scope_name, shots=None)
print(data_arr.shape, tarr.shape)

(610, 250001) (250001,)


In [6]:
plt.figure()
plt.plot(tarr, data_arr[0,:])